<!--
File: notebooks_v2/01_dataset_creation_v2.ipynb
What does this file do?
    Explains and inspects the Nepal-first V2 dataset creation workflow.
Methods/functions this file contains:
    Notebook cells for loading generated CSV files, checking schema, and summarizing dataset quality.
Date and Day of last modification:
    2026-05-21, Thursday.
-->

# 01 - Nepal Finance Dataset Creation V2

This notebook documents the dataset side of the project. The V2 dataset is Nepal-first, NPR-normalized, multi-currency aware, and includes personal plus shared project expenses.

Main generated folder: `output_v2/`

## Dataset Design

The dataset includes:

- users and financial personas
- income events
- budgets and goals
- personal expenses
- project/shared expenses
- expense splits
- recurring payments
- currency rates
- festival calendar

The aim is not just to predict expenses, but to provide financial assistance: budget risk, unusual spending, category focus, and shared-settlement insight.

In [ ]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks_v2' else Path.cwd()
DATA_DIR = PROJECT_ROOT / 'output_v2'
files = sorted(p.name for p in DATA_DIR.glob('*.csv'))
files

In [ ]:
frames = {path.stem: pd.read_csv(path, low_memory=False) for path in DATA_DIR.glob('*.csv')}
summary = pd.DataFrame([
    {'table': name, 'rows': len(df), 'columns': len(df.columns)}
    for name, df in frames.items()
]).sort_values('table')
summary

In [ ]:
expenses = frames['expenses'].copy()
expenses['date'] = pd.to_datetime(expenses['date'])

print('Users:', len(frames['users']))
print('Expenses:', len(expenses))
print('Shared expenses:', len(frames['shared_expenses']))
print('Expense splits:', len(frames['expense_splits']))
print('Date range:', expenses['date'].min().date(), 'to', expenses['date'].max().date())
print('Currencies:', expenses['currency_code'].value_counts().to_dict())

In [ ]:
category_summary = expenses.groupby('category').agg(
    rows=('expense_id', 'count'),
    total_npr=('amount_npr', 'sum'),
    median_npr=('amount_npr', 'median'),
).sort_values('total_npr', ascending=False)
category_summary.head(12)

In [ ]:
users = frames['users']
users['financial_persona'].value_counts(normalize=True).mul(100).round(2)